# Part 2: Baseline Econometrics & Event Studies

Having constructed our panel dataset, this notebook implements our core empirical strategy: a Two-Way Fixed Effects (TWFE) Difference-in-Differences model to quantify the impact of the Kulturkampf on fertility outcomes in Catholic areas.


### 1. Setup and Data Loading
We load the pre-compiled `.parquet` panel dataset from Part 1.


In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Enable autoreload for development
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Paths
DATA_RAW = project_root / "data" / "raw"
DATA_PROCESSED = project_root / "data" / "processed"
OUTPUTS = project_root / "outputs" / "figures"
OUTPUTS.mkdir(exist_ok=True, parents=True)

print("Setup complete. Outputs will be saved to:", OUTPUTS)

from src.analysis.regressions import run_baseline_did, run_event_study, run_robustness
from src.visualization.plots import plot_event_study, plot_robustness_table
from src.data.load_data import load_ipehd_master, load_rel1871

# Load pre-processed panel
panel = pd.read_parquet(DATA_PROCESSED / "analysis_panel.parquet")



### 2. Baseline Difference-in-Differences
Our baseline specification relies on TWFE: $Y_{it} = \beta (CathShare_i \times Post_t) + \alpha_i + \delta_t + X_{it}'\gamma + \varepsilon_{it}$. We test both continuous treatment intensity (Catholic share percentage) and a binary treatment indicator (>50% Catholic).


In [ ]:
print("=" * 60)
print("BASELINE DiD: CBR ~ CathShare × Post")
print("=" * 60)
res_cont = run_baseline_did(panel, outcome="cbr", treatment="continuous")
print(res_cont["summary"])

print("=" * 60)
print("BASELINE DiD: CBR ~ HighCath × Post")
print("=" * 60)
res_bin = run_baseline_did(panel, outcome="cbr", treatment="binary")
print(res_bin["summary"])



We also check if this effect persists across other demographic outcomes, such as legitimate births and marriage rates. A change in crude birth rate could be mechanically driven by fewer marriages rather than altered marital fertility.


In [ ]:
print("=" * 60)
print("DiD FOR ALTERNATIVE OUTCOMES")
print("=" * 60)
for outcome, label in [
    ("legitimate_br", "Legitimate birth rate"),
    ("marriage_rate", "Marriage rate"),
    ("illegitimacy_ratio", "Illegitimacy ratio"),
]:
    print(f"\n--- {label} ---")
    try:
        res = run_baseline_did(panel, outcome=outcome, treatment="continuous")
        r = res["result"]
        treat_var = "cath_share_x_post"
        print(f"  Coef: {r.params[treat_var]:.4f} (SE: {r.std_errors[treat_var]:.4f}, p: {r.pvalues[treat_var]:.3f})")
    except Exception as e:
        print(f"  Error: {e}")



### 3. Event Study Design
A fundamental assumption of DiD is parallel trends. We plot an event study interacting the Catholic share with year dummies to trace the dynamic effect over time and formally test for pre-existing trends prior to 1872.


In [ ]:
print("=" * 60)
print("EVENT STUDY")
print("=" * 60)

es = run_event_study(panel, outcome="cbr", treatment_var="cath_share", ref_year=1872)
fig, ax = plot_event_study(
    es["coefs"],
    ref_year=1872,
    title="Event study: Catholic share × Year dummies on CBR",
    savepath=str(OUTPUTS / "fig5_event_study.png"),
)
plt.show()

# Formal pre-trend test
pre_coefs = es["coefs"][es["coefs"]["Year"] < 1872]
print(f"\nPre-trend coefficients (before 1872):")
print(pre_coefs[["Year", "beta", "se"]].to_string(index=False))



### 4. Robustness Checks
To verify that our results are not artifacts of arbitrary methodological choices, we test alternative temporal cutoffs (1872, 1875), alternative treatment thresholds, and the exclusion of specific demographics like Polish-majority provinces.


In [ ]:
print("=" * 60)
print("ROBUSTNESS CHECKS")
print("=" * 60)
rob = run_robustness(panel, outcome="cbr")
fig, ax = plot_robustness_table(rob, savepath=str(OUTPUTS / "fig6_robustness.png"))
plt.show()



### 5. Validation against iPEHD
Finally, we cross-validate our key treatment variable (Catholic share) against the authoritative Becker-Woessmann (2009) iPEHD dataset to ensure our REL1871 data extraction is robust.


In [ ]:
ipehd = load_ipehd_master(DATA_RAW / "ipehd_qje2009_master.dta")
rel = load_rel1871(DATA_RAW / "REL1871.XLS")

print(f"iPEHD f_cath: mean={ipehd['f_cath'].mean():.1f}, median={ipehd['f_cath'].median():.1f}, N={len(ipehd)}")
print(f"Galloway cath_share: mean={rel['cath_share'].mean():.1f}, median={rel['cath_share'].median():.1f}, N={len(rel)}")

